# 第 13 天：流动性因子

> 来自《30 天因子研究计划》第 13 天  
> 主题：流动性因子  
> 必做：换手率  
> 选做：Amihud  
> 目标产出：流动性因子库

---

## 0. 今天你要真正学会什么？

流动性因子关注的是：


一只股票是否容易交易？


今天掌握：

1. 换手率如何计算。
2. 成交额为什么能衡量流动性。
3. Amihud 非流动性指标怎么计算。
4. 如何构建流动性因子库。

---

## 1. 换手率直觉

换手率：


turnover = volume / shares_outstanding


它表示一天交易的股票数量占总股本的比例。

- 换手高：交易活跃。
- 换手低：交易冷清。

---

## 2. Amihud 指标

Amihud 非流动性：


Amihud = mean(abs(return) / amount)


直觉：

> 同样的成交额，如果价格波动更大，说明冲击更强，流动性更差。

Amihud 越高，通常代表越不流动。

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260706)


---

## 4. 构造模拟成交数据


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=220)
tickers = [f"Stock_{i:03d}" for i in range(100)]

close = pd.DataFrame(index=dates, columns=tickers, dtype=float)
volume = pd.DataFrame(index=dates, columns=tickers, dtype=float)
shares = pd.Series(index=tickers, dtype=float)

for ticker in tickers:
    shares[ticker] = rng.lognormal(mean=18.5, sigma=0.8)
    ret = rng.normal(0.0002, rng.uniform(0.012, 0.03), len(dates))
    close[ticker] = 30 * np.cumprod(1 + ret)
    volume[ticker] = rng.lognormal(mean=13.5, sigma=0.9, size=len(dates))

amount = close * volume
close.head()


---

## 5. 计算换手率


In [ ]:
turnover = volume.divide(shares, axis=1)
turnover_20d = turnover.rolling(20).mean()
amount_20d = amount.rolling(20).mean()

turnover_20d.tail()


---

## 6. 计算 Amihud


In [ ]:
daily_ret = close.pct_change()
amihud_daily = daily_ret.abs() / amount.replace(0, np.nan)
amihud_20d = amihud_daily.rolling(20).mean()

amihud_20d.tail()


因为 Amihud 越高越不流动，如果想让因子越大代表越流动，可以取负号：


In [ ]:
liquidity_amihud = -amihud_20d


---

## 7. 构建流动性因子库


In [ ]:
def wide_to_long(wide: pd.DataFrame, name: str) -> pd.DataFrame:
    return (
        wide.stack(future_stack=True)
        .dropna()
        .rename(name)
        .reset_index()
        .rename(columns={"level_0": "date", "level_1": "ticker"})
    )


liquidity_library = wide_to_long(turnover_20d, "turnover_20d")
liquidity_library = liquidity_library.merge(wide_to_long(amount_20d, "amount_20d"), on=["date", "ticker"], how="outer")
liquidity_library = liquidity_library.merge(wide_to_long(amihud_20d, "amihud_20d"), on=["date", "ticker"], how="outer")
liquidity_library["liquidity_amihud"] = -liquidity_library["amihud_20d"]
liquidity_library.head()


---

## 8. 快速检验


In [ ]:
future_20d_ret = close.shift(-20) / close - 1
label = wide_to_long(future_20d_ret, "future_20d_ret")
data = liquidity_library.merge(label, on=["date", "ticker"], how="inner").dropna()

daily_ic = data.groupby("date").apply(
    lambda g: g["liquidity_amihud"].corr(g["future_20d_ret"], method="spearman"),
    include_groups=False
)

{"mean_rank_ic": daily_ic.mean(), "icir": daily_ic.mean() / daily_ic.std(), "positive_ratio": (daily_ic > 0).mean()}


---

## 9. 目标产出：流动性因子库函数


In [ ]:
def build_liquidity_factor_library(close: pd.DataFrame, volume: pd.DataFrame, shares: pd.Series) -> pd.DataFrame:
    amount = close * volume
    turnover = volume.divide(shares, axis=1)
    daily_ret = close.pct_change()
    amihud = (daily_ret.abs() / amount.replace(0, np.nan)).rolling(20).mean()

    out = wide_to_long(turnover.rolling(20).mean(), "turnover_20d")
    out = out.merge(wide_to_long(amount.rolling(20).mean(), "amount_20d"), on=["date", "ticker"], how="outer")
    out = out.merge(wide_to_long(amihud, "amihud_20d"), on=["date", "ticker"], how="outer")
    out["liquidity_amihud"] = -out["amihud_20d"]
    return out


liquidity_factor_library = build_liquidity_factor_library(close, volume, shares)
liquidity_factor_library.tail()


---

## 10. 知识图谱


In [ ]:
mindmap
  root((流动性因子))
    换手率
      成交量除以股本
      活跃程度
    成交额
      价格乘成交量
      容量
    Amihud
      收益冲击除以成交额
      越高越不流动
    风险
      停牌
      涨跌停
      冲击成本


---

## 11. 作业

1. 比较换手率和成交额的相关性。
2. 把 Amihud 窗口改成 60 日。
3. 检查低流动性股票是否更高波动。
4. 思考流动性和交易成本的关系。

---

## 12. 自测题

1. 换手率公式是什么？  
   答案：成交量 / 总股本或流通股本。

2. Amihud 越高代表什么？  
   答案：通常代表越不流动。

3. 为什么流动性因子很重要？  
   答案：它影响收益解释、交易成本和策略容量。

---

## 13. 明天预告

明天学习因子中性化：行业中性化和市值中性化。

---

## 14. 仅供学习的提醒

本文使用模拟数据解释流动性因子，不构成任何投资建议。

---

# 统一高质量增强模块

> 本增强模块用于把第 13 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：流动性因子
- 必做：换手率
- 选做：Amihud
- 目标产出：流动性因子库

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

一个策略回测赚钱，但买不进去、卖不出来，或者一交易就把价格打穿，这就是流动性问题。

这个例子背后的关键直觉是：

> 能不能交易、以什么成本交易，是因子能否落地的核心。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


流动性因子
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 流动性因子库


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(113)
dates = pd.bdate_range("2024-01-02", periods=160)
close = pd.Series(30 * np.cumprod(1 + rng.normal(.0002, .02, len(dates))), index=dates)
volume = pd.Series(rng.lognormal(13.5, .8, len(dates)), index=dates)
shares = rng.lognormal(18, .5)
amount = close * volume
turnover20 = (volume / shares).rolling(20).mean()
amihud20 = (close.pct_change().abs() / amount).rolling(20).mean()
print(pd.DataFrame({"turnover20": turnover20, "amihud20": amihud20}).tail())


## E. 产出验收标准

完成今天课程后，你的 `流动性因子库` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `流动性因子` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `流动性因子库` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `流动性因子库`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 13 天复盘：流动性因子

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
---

# 第 10-15 天深度加厚模块


    ## L. 为什么还要加厚这一课？

    这一课属于第 10-15 天的“工程化因子”部分：它不像 Alpha、Beta 那样只靠概念就能建立直觉，也不像 PE、ROE 那样有明确财务含义。它更依赖窗口、参数、预处理顺序和检验口径。

    所以学习 `流动性因子` 时，不能只停留在“知道公式”。你至少要完成三层理解：


    第一层：公式能写对
    第二层：参数变化后结果还能解释
    第三层：能放进统一因子流水线


    如果只学第一层，代码很快能写出来，但研究时很容易陷入“参数换一下结果就变”的困境。

    ## M. 更贴近真实研究的场景

    一个因子在小票里很赚钱，但那些股票成交额很低、Amihud 很高。回测净值很好看，真实下单却可能没有容量。

    这类问题在真实研究中很常见：一个因子看似简单，但只要换股票池、换窗口、换持有期、换市场阶段，结果就会明显变化。成熟的研究方式不是逃避这种变化，而是把变化记录下来、解释出来。

    ## N. 参数敏感性实验

    下面这段代码是专门为本课补充的参数实验。它不追求复杂，而是训练一个习惯：

    > 不要只交一个因子结果，至少比较几组合理参数。


In [ ]:
    import numpy as np
import pandas as pd

rng = np.random.default_rng(213)
n = 300
df = pd.DataFrame({
    "ticker": [f"S{i:03d}" for i in range(n)],
    "daily_ret_abs": rng.lognormal(-4.2, .7, n),
    "amount": rng.lognormal(15, 1.2, n),
    "volume": rng.lognormal(13, 1.0, n),
    "shares": rng.lognormal(18, .8, n),
})
df["turnover"] = df["volume"] / df["shares"]
df["amihud"] = df["daily_ret_abs"] / df["amount"]
df["liquidity_score"] = df["turnover"].rank(pct=True) - df["amihud"].rank(pct=True)
print(df[["turnover", "amihud", "liquidity_score"]].describe().round(6))
print(df.sort_values("liquidity_score").head()[["ticker","turnover","amihud","liquidity_score"]])


    ## O. 结果该怎么写进研究笔记？

    建议你用下面这个格式记录：


    因子名称：流动性因子

    1. 使用的数据：
       - 股票池：
       - 时间区间：
       - 价格 / 财务口径：

    2. 核心参数：
       - 主参数：
       - 对照参数：

    3. 因子方向：
       - 因子值越大代表：
       - 是否需要取负号：

    4. 检验结果：
       - Rank IC：
       - ICIR：
       - 分组收益：
       - 多空表现：

    5. 稳定性：
       - 参数变化后是否稳定：
       - 分阶段是否稳定：
       - 极端行情是否失效：

    6. 结论：
       - 是否进入因子库：
       - 还需要什么后续验证：


    ## P. 额外验收清单

    `流动性因子库` 如果要达到可复用标准，额外检查：

    1. 同时看换手率、成交额和 Amihud。
2. 明确 Amihud 越高越不流动。
3. 不要忽略停牌、涨跌停和冲击成本。
4. 报告低流动性样本占比。

    ## Q. 更深入的常见误区

    ### 误区 1：参数越多越高级

    参数多不代表研究深，很多时候只是过拟合空间更大。真正高级的是解释参数为什么合理，并证明它在相邻参数下仍然不崩。

    ### 误区 2：只看全样本平均

    全样本平均可能掩盖阶段失效。至少要分年度、分市场状态、分股票池看一次。

    ### 误区 3：把预处理当成机械步骤

    去极值、标准化、中性化会改变因子含义。每加一步，都要知道自己剥离了什么，也可能损失了什么。

    ### 误区 4：忽略交易可行性

    技术、波动率、流动性类因子往往换手更高，交易成本可能非常关键。纸面有效不等于可交易。

    ## R. 加厚作业

    1. 把本课主参数上下各调整一次，记录结果变化。
    2. 把未来收益标签从 20 日改成 5 日和 60 日，观察结论是否变。
    3. 随机删除 10% 股票样本，检查结果是否稳定。
    4. 把因子取反，确认分组结果是否镜像变化。
    5. 写一段 200 字研究结论，必须同时包含“支持证据”和“风险提示”。

    ## S. 一句话升级结论

    `流动性因子` 的高质量学习标准不是“会算”，而是：

    > 会定义、会检验、会解释参数变化，也知道它在真实交易里可能被什么击穿。
